# Part 7: The same model, twice

You now have the same network in both frameworks. This notebook answers two questions. Are they really the same model? And do they give the same results when you train them?

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow start-up messages

import numpy as np
import torch

from churn_data import make_splits
from churn_net import ChurnNet
from compare import copy_torch_into_keras
from keras_net import build_keras_net

splits = make_splits(42)
x_val = splits.val.x

## Same architecture, same weights, same predictions

Build both networks and copy the PyTorch weights into Keras. A Keras kernel has shape `(inputs, outputs)` and a PyTorch weight has shape `(outputs, inputs)`, so the copy transposes.

In [ ]:
torch.manual_seed(0)
torch_net = ChurnNet(9).eval()
keras_net = build_keras_net(9)

print("PyTorch parameters:", sum(p.numel() for p in torch_net.parameters()))
print("Keras parameters:  ", keras_net.count_params())

copy_torch_into_keras(torch_net, keras_net)
with torch.no_grad():
    from_torch = torch_net(x_val).numpy()
from_keras = keras_net.predict(x_val.numpy(), verbose=0).ravel()
print("largest difference in predictions:", float(np.abs(from_torch - from_keras).max()))

Both networks hold 449 numbers, and with the same numbers they agree to about eight decimal places. A framework is a way of writing and training a model, and the model itself is the same.

## Different starting points

When each framework picks its own starting numbers, they differ, because the defaults are not the same.

In [ ]:
torch.manual_seed(0)
fresh_torch = ChurnNet(9)
fresh_keras = build_keras_net(9)
w_torch = fresh_torch.layers[0].weight.detach().numpy()
w_keras, b_keras = fresh_keras.layers[0].get_weights()
print("first-layer weights, PyTorch:", w_torch.min().round(2), "to", w_torch.max().round(2))
print("first-layer weights, Keras:  ", w_keras.min().round(2), "to", w_keras.max().round(2))
print("first-layer biases, PyTorch: ", fresh_torch.layers[0].bias[:3].detach().numpy().round(2))
print("first-layer biases, Keras:   ", b_keras[:3])

PyTorch draws weights from a narrower range and gives the biases small random values. Keras uses Glorot uniform weights and starts all biases at zero. Add different shuffling of the batches and you cannot expect identical training runs. What you can expect is similar results.

## Training both, on the same data

`compare.py` trains both with the same split, the same learning rate and batch size, the same class weighting, and keeps the best epoch by validation loss. It runs three seeds, so the table below takes about a minute.

In [ ]:
from compare import main

main()

Read the table before you decide anything:

- Both frameworks land close together. Keras is a little higher on all three seeds (about 0.36 against 0.34), but with only 400 test rows and 57 churners a gap of a few hundredths in F1 is inside the noise, so this table does not name a winner on accuracy.
- The best epoch differs by a lot. That is the initialization and shuffling at work.
- PyTorch ran this tiny model several times faster. `fit` carries fixed overhead for every epoch, so the gap shrinks as the model grows, and it should not decide your choice.

One trap sits in this comparison. Keras's `class_weight` changes only the **training** loss. Its validation loss is unweighted unless you pass weights as the third item of `validation_data`. The loss that picks the best epoch then means something different from PyTorch's, and the Keras F1 drops to about 0.28 to 0.31. When two frameworks disagree, first check that they are measuring the same thing.

## Your turn

Run `main()` again with `SEEDS = (3, 4, 5)` in `compare.py`. Do the conclusions above still hold?